# T20 - 债券条款分析

## 项目概述

本项目专注于**信用债加速到期条款**的研究和分析，帮助投资者了解债券条款中的风险触发条件。

### 核心功能
- 加速到期条款识别
- 触发条件分析
- 条款可视化
- 个券条款统计

### 应用场景
- 债券投资风险评估
- 条款比较分析
- 信用风险预警

---

## 触发条件分类

| 条件类型 | 关键词/模式 |
|----------|-------------|
| 交叉违约 | 交叉违约、子公司违约、贷款违约 |
| 募投项目问题 | 项目未开工、项目终止 |
| 未能偿付本息 | 未能偿付本金、利息逾期 |
| 违反契约承诺 | 违反约定、不履行承诺 |
| 丧失清偿能力 | 破产、清算、接管 |
| 重大资产变动 | 出售重大资产、资产重组 |
| 注销解散 | 解散、吊销营业执照 |
| 实际控制人变更 | 控制权变更 |
| 评级下调 | 信用评级下调 |
| 财务指标恶化 | 资产负债率超标 |

---

## 1. 环境配置

导入必要的依赖库，并加载配置参数。

In [ ]:
# 标准库
import os
import re
import json
from collections import Counter

# 第三方库
import pandas as pd
import numpy as np

# 配置文件
from config import (
    DATA_CONFIG,
    TRIGGER_PATTERNS,
    PROTECTION_MECHANISM_PATTERNS,
    OTHER_FEATURE_PATTERNS,
    REPAYMENT_SCHEDULE_PATTERNS,
    CROSS_DEFAULT_THRESHOLD_PATTERNS,
    ANALYSIS_CONFIG,
    VISUALIZATION_CONFIG
)

print("环境配置完成")
print(f"工作目录: {os.getcwd()}")

In [ ]:
# 创建输出目录
output_dir = DATA_CONFIG["output_dir"]
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"创建输出目录: {output_dir}")
else:
    print(f"输出目录已存在: {output_dir}")

---

## 2. 数据获取

读取债券条款数据，支持Excel格式输入。

In [ ]:
def read_excel_file(file_path):
    """
    读取Excel文件内容
    
    参数:
        file_path: Excel文件路径
    
    返回:
        DataFrame或None
    """
    try:
        df = pd.read_excel(file_path)
        print(f"成功读取Excel文件，包含 {len(df)} 行数据")
        print(f"列名: {df.columns.tolist()}")
        return df
    except FileNotFoundError:
        print(f"文件未找到: {file_path}")
        return None
    except Exception as e:
        print(f"读取Excel文件时出错: {e}")
        return None

In [ ]:
# 读取数据文件
input_path = DATA_CONFIG["input_excel_path"]

# 如果输入文件不存在，创建示例数据
if not os.path.exists(input_path):
    print(f"输入文件不存在: {input_path}")
    print("创建示例数据用于演示...")
    
    # 创建示例数据
    sample_data = pd.DataFrame({
        '债券代码': ['123456.SH', '234567.SH', '345678.SH'],
        '债券名称': ['示例债券1', '示例债券2', '示例债券3'],
        '加速到期条款说明': [
            '若发行人发生交叉违约事件，债券持有人有权要求提前清偿全部本金。发行人应在3个工作日内书面通知持有人。',
            '若发行人丧失清偿能力或进入破产程序，本金和利息视为立即到期。债券持有人会议可决定增加担保措施。',
            '若募投项目未开工或提前终止，投资者有权选择回售。宽限期内予以纠正完毕的除外。'
        ]
    })
    
    # 创建数据目录
    os.makedirs(os.path.dirname(input_path), exist_ok=True)
    sample_data.to_excel(input_path, index=False)
    print(f"示例数据已保存到: {input_path}")
    
    df = sample_data
else:
    df = read_excel_file(input_path)

if df is not None:
    print(f"\n数据预览:")
    display(df.head())

---

## 3. 数据处理

从DataFrame中提取债券信息，并进行数据清洗和预处理。

In [ ]:
def extract_bonds_from_df(df):
    """
    从DataFrame中提取债券信息
    
    参数:
        df: 包含债券数据的DataFrame
    
    返回:
        债券信息列表
    """
    bonds = []
    
    # 确定列名（不区分大小写和空格）
    code_col = next((col for col in df.columns if '代码' in str(col)), None)
    name_col = next((col for col in df.columns if '名称' in str(col)), None)
    clause_col = next((col for col in df.columns if '条款' in str(col) or '说明' in str(col)), None)
    
    if not all([code_col, name_col, clause_col]):
        print(f"无法识别所需列名。已找到的列: {df.columns.tolist()}")
        return []
    
    print(f"使用列: 代码={code_col}, 名称={name_col}, 条款={clause_col}")
    
    # 遍历每行提取信息
    for _, row in df.iterrows():
        code = str(row[code_col]).strip()
        name = str(row[name_col]).strip()
        clause = str(row[clause_col]).strip()
        
        # 跳过空行或缺失关键信息的行
        if pd.isna(code) or pd.isna(clause) or not code or code == 'nan':
            continue
            
        bonds.append({
            'code': code,
            'name': name,
            'clause': clause
        })
    
    return bonds

In [ ]:
# 提取债券信息
bonds = extract_bonds_from_df(df)

if bonds:
    print(f"成功提取 {len(bonds)} 只带加速到期条款的债券")
    print(f"\n示例债券:")
    for bond in bonds[:3]:
        print(f"  - {bond['code']}: {bond['name']}")
else:
    print("未能提取到债券信息")

---

## 4. 核心逻辑

包含条款识别、上下文提取和统计分析的核心算法。

### 4.1 加速到期触发条件分析

In [ ]:
def analyze_acceleration_triggers(bonds, trigger_patterns=None, max_examples=3, context_length=50):
    """
    分析加速到期条款的触发条件
    
    参数:
        bonds: 债券信息列表
        trigger_patterns: 触发条件正则表达式模式字典
        max_examples: 每类触发条件最多保存的示例数量
        context_length: 上下文提取长度
    
    返回:
        (trigger_stats, trigger_examples) 元组
    """
    if trigger_patterns is None:
        trigger_patterns = TRIGGER_PATTERNS
    
    trigger_stats = {key: 0 for key in trigger_patterns}
    trigger_examples = {key: [] for key in trigger_patterns}
    
    for bond in bonds:
        code = bond['code']
        clause = bond['clause']
        for trigger, patterns in trigger_patterns.items():
            for pattern in patterns:
                if re.search(pattern, clause):
                    trigger_stats[trigger] += 1
                    # 保存示例
                    if len(trigger_examples[trigger]) < max_examples:
                        match = re.search(pattern, clause)
                        start = max(0, match.start() - context_length)
                        end = min(len(clause), match.end() + context_length)
                        context = f"{clause[start:end]}..."
                        trigger_examples[trigger].append({
                            'code': code,
                            'context': context
                        })
                    break
    
    return trigger_stats, trigger_examples

In [ ]:
# 执行触发条件分析
trigger_stats, trigger_examples = analyze_acceleration_triggers(
    bonds,
    max_examples=ANALYSIS_CONFIG["max_examples_per_trigger"],
    context_length=ANALYSIS_CONFIG["context_length"]
)

print("加速到期触发条件统计:")
for trigger, count in sorted(trigger_stats.items(), key=lambda x: x[1], reverse=True):
    pct = count / len(bonds) * 100 if bonds else 0
    print(f"  {trigger}: {count} 只债券 ({pct:.1f}%)")

### 4.2 投资者保护机制分析

In [ ]:
def analyze_protection_mechanisms(bonds, mechanism_patterns=None, max_examples=3, context_length=50):
    """
    分析投资者保护机制
    
    参数:
        bonds: 债券信息列表
        mechanism_patterns: 保护机制正则表达式模式字典
        max_examples: 每类保护机制最多保存的示例数量
        context_length: 上下文提取长度
    
    返回:
        (mechanisms, mechanism_examples) 元组
    """
    if mechanism_patterns is None:
        mechanism_patterns = PROTECTION_MECHANISM_PATTERNS
    
    mechanisms = {key: 0 for key in mechanism_patterns}
    mechanism_examples = {key: [] for key in mechanism_patterns}
    
    for bond in bonds:
        code = bond['code']
        clause = bond['clause']
        
        for mechanism, patterns in mechanism_patterns.items():
            for pattern in patterns:
                if re.search(pattern, clause):
                    mechanisms[mechanism] += 1
                    if len(mechanism_examples[mechanism]) < max_examples:
                        match = re.search(pattern, clause)
                        start = max(0, match.start() - context_length)
                        end = min(len(clause), match.end() + context_length)
                        context = f"{clause[start:end]}..."
                        mechanism_examples[mechanism].append({
                            'code': code,
                            'context': context
                        })
                    break
    
    return mechanisms, mechanism_examples

In [ ]:
# 执行保护机制分析
mechanism_stats, mechanism_examples = analyze_protection_mechanisms(
    bonds,
    max_examples=ANALYSIS_CONFIG["max_examples_per_mechanism"],
    context_length=ANALYSIS_CONFIG["context_length"]
)

print("投资者保护机制统计:")
for mechanism, count in sorted(mechanism_stats.items(), key=lambda x: x[1], reverse=True):
    pct = count / len(bonds) * 100 if bonds else 0
    print(f"  {mechanism}: {count} 只债券 ({pct:.1f}%)")

### 4.3 其他债券特征分析

In [ ]:
def analyze_other_features(bonds, feature_patterns=None, max_examples=3, context_length=50):
    """
    分析其他常见债券特征
    
    参数:
        bonds: 债券信息列表
        feature_patterns: 特征正则表达式模式字典
        max_examples: 每类特征最多保存的示例数量
        context_length: 上下文提取长度
    
    返回:
        (features, feature_examples) 元组
    """
    if feature_patterns is None:
        feature_patterns = OTHER_FEATURE_PATTERNS
    
    features = {key: 0 for key in feature_patterns}
    feature_examples = {key: [] for key in feature_patterns}
    
    for bond in bonds:
        code = bond['code']
        clause = bond['clause']
        
        for feature, patterns in feature_patterns.items():
            for pattern in patterns:
                if re.search(pattern, clause):
                    features[feature] += 1
                    if len(feature_examples[feature]) < max_examples:
                        match = re.search(pattern, clause)
                        start = max(0, match.start() - context_length)
                        end = min(len(clause), match.end() + context_length)
                        context = f"{clause[start:end]}..."
                        feature_examples[feature].append({
                            'code': code,
                            'context': context
                        })
                    break
    
    return features, feature_examples

In [ ]:
# 执行其他特征分析
feature_stats, feature_examples = analyze_other_features(
    bonds,
    max_examples=ANALYSIS_CONFIG["max_examples_per_feature"],
    context_length=ANALYSIS_CONFIG["context_length"]
)

print("其他债券特征统计:")
for feature, count in sorted(feature_stats.items(), key=lambda x: x[1], reverse=True):
    pct = count / len(bonds) * 100 if bonds else 0
    print(f"  {feature}: {count} 只债券 ({pct:.1f}%)")

### 4.4 分期还本比例模式分析

In [ ]:
def extract_repayment_schedule(bonds, schedule_patterns=None, max_examples=2):
    """
    分析分期还本的具体比例模式
    
    参数:
        bonds: 债券信息列表
        schedule_patterns: 分期还本正则表达式模式列表
        max_examples: 每种模式最多保存的示例数量
    
    返回:
        分期还本模式字典
    """
    if schedule_patterns is None:
        schedule_patterns = REPAYMENT_SCHEDULE_PATTERNS
    
    repayment_patterns = {}
    
    for bond in bonds:
        clause = bond['clause']
        
        for pattern in schedule_patterns:
            match = re.search(pattern, clause)
            if match:
                groups = match.groups()
                if len(groups) >= 5:
                    # 处理年份和比例
                    if len(groups) >= 10:  # 包含年份和比例
                        years = groups[:5]
                        rates = groups[5:10]
                    else:  # 只有比例，没有年份
                        years = ['?', '?', '?', '?', '?']
                        rates = groups[:5]
                    
                    pattern_key = "-".join(rates)
                    if pattern_key not in repayment_patterns:
                        repayment_patterns[pattern_key] = {
                            'count': 0,
                            'years': years,
                            'rates': rates,
                            'examples': []
                        }
                    repayment_patterns[pattern_key]['count'] += 1
                    if len(repayment_patterns[pattern_key]['examples']) < max_examples:
                        repayment_patterns[pattern_key]['examples'].append(bond['code'])
                break
    
    return repayment_patterns

In [ ]:
# 执行分期还本分析
repayment_patterns = extract_repayment_schedule(
    bonds,
    max_examples=ANALYSIS_CONFIG["max_examples_per_pattern"]
)

if repayment_patterns:
    print("分期还本比例模式统计:")
    for pattern, data in sorted(repayment_patterns.items(), key=lambda x: x[1]['count'], reverse=True):
        pct = data['count'] / len(bonds) * 100 if bonds else 0
        print(f"  模式 {pattern}: {data['count']} 只债券 ({pct:.1f}%)")
        print(f"    年份: 第{', 第'.join(data['years'])}年末")
        print(f"    比例: {'-'.join(data['rates'])}%")
        print(f"    示例债券: {', '.join(data['examples'])}")
else:
    print("未发现分期还本模式")

### 4.5 交叉违约金额门槛分析

In [ ]:
def extract_crossdefault_thresholds(bonds, threshold_patterns=None, max_examples=2):
    """
    分析交叉违约的金额门槛
    
    参数:
        bonds: 债券信息列表
        threshold_patterns: 金额门槛正则表达式模式列表
        max_examples: 每种门槛最多保存的示例数量
    
    返回:
        门槛字典
    """
    if threshold_patterns is None:
        threshold_patterns = CROSS_DEFAULT_THRESHOLD_PATTERNS
    
    thresholds = {}
    
    for bond in bonds:
        clause = bond['clause']
        
        for pattern in threshold_patterns:
            match = re.search(pattern, clause)
            if match:
                # 提取金额并规范化（移除逗号）
                amount = match.group(1).replace(',', '').replace('，', '')
                threshold = f"{amount}万元"
                if threshold not in thresholds:
                    thresholds[threshold] = {
                        'count': 0,
                        'examples': []
                    }
                thresholds[threshold]['count'] += 1
                if len(thresholds[threshold]['examples']) < max_examples:
                    thresholds[threshold]['examples'].append(bond['code'])
                break
    
    return thresholds

In [ ]:
# 执行交叉违约门槛分析
threshold_patterns = extract_crossdefault_thresholds(
    bonds,
    max_examples=ANALYSIS_CONFIG["max_examples_per_threshold"]
)

if threshold_patterns:
    print("交叉违约金额门槛统计:")
    for threshold, data in sorted(threshold_patterns.items(), key=lambda x: x[1]['count'], reverse=True):
        pct = data['count'] / len(bonds) * 100 if bonds else 0
        print(f"  门槛 {threshold}: {data['count']} 只债券 ({pct:.1f}%)")
        print(f"    示例债券: {', '.join(data['examples'])}")
else:
    print("未发现交叉违约金额门槛")

---

## 5. 执行与测试

运行主流程，生成分析结果并导出报告。

In [ ]:
# 汇总分析结果
print("=" * 60)
print("债券条款分析报告摘要")
print("=" * 60)
print(f"分析债券总数: {len(bonds)}")
print()

print("【触发条件统计】")
for trigger, count in sorted(trigger_stats.items(), key=lambda x: x[1], reverse=True):
    if count > 0:
        pct = count / len(bonds) * 100 if bonds else 0
        print(f"  {trigger}: {count} ({pct:.1f}%)")

print()
print("【保护机制统计】")
for mechanism, count in sorted(mechanism_stats.items(), key=lambda x: x[1], reverse=True):
    if count > 0:
        pct = count / len(bonds) * 100 if bonds else 0
        print(f"  {mechanism}: {count} ({pct:.1f}%)")

print()
print("【其他特征统计】")
for feature, count in sorted(feature_stats.items(), key=lambda x: x[1], reverse=True):
    if count > 0:
        pct = count / len(bonds) * 100 if bonds else 0
        print(f"  {feature}: {count} ({pct:.1f}%)")

In [ ]:
# 导出结果到CSV
df_results = pd.DataFrame([
    {"类别": "触发条件", "特征": k, "数量": v, 
     "占比": f"{v/len(bonds)*100:.1f}%" if bonds else "0%",
     "示例": '; '.join([f"{ex['code']}: {ex['context'][:50]}..." for ex in trigger_examples[k][:1]])} 
    for k, v in trigger_stats.items()
] + [
    {"类别": "保护机制", "特征": k, "数量": v,
     "占比": f"{v/len(bonds)*100:.1f}%" if bonds else "0%",
     "示例": '; '.join([f"{ex['code']}: {ex['context'][:50]}..." for ex in mechanism_examples[k][:1]])} 
    for k, v in mechanism_stats.items()
] + [
    {"类别": "其他特征", "特征": k, "数量": v,
     "占比": f"{v/len(bonds)*100:.1f}%" if bonds else "0%",
     "示例": '; '.join([f"{ex['code']}: {ex['context'][:50]}..." for ex in feature_examples[k][:1]])} 
    for k, v in feature_stats.items()
])

# 保存CSV
csv_path = os.path.join(output_dir, DATA_CONFIG["output_csv"])
df_results.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"分析结果已保存到: {csv_path}")

In [ ]:
# 保存详细分析结果到JSON
detailed_results = {
    "债券总数": len(bonds),
    "加速到期触发条件": {
        "统计": trigger_stats,
        "示例": {k: [{"代码": ex["code"], "内容": ex["context"]} for ex in v] 
                for k, v in trigger_examples.items()}
    },
    "投资者保护机制": {
        "统计": mechanism_stats,
        "示例": {k: [{"代码": ex["code"], "内容": ex["context"]} for ex in v] 
                for k, v in mechanism_examples.items()}
    },
    "其他债券特征": {
        "统计": feature_stats,
        "示例": {k: [{"代码": ex["code"], "内容": ex["context"]} for ex in v] 
                for k, v in feature_examples.items()}
    },
    "分期还本模式": {
        pattern: {
            "数量": data["count"],
            "占比": f"{data['count']/len(bonds)*100:.1f}%" if bonds else "0%",
            "年份": list(data["years"]),
            "比例": list(data["rates"]),
            "示例": data["examples"]
        } for pattern, data in repayment_patterns.items()
    },
    "交叉违约金额门槛": {
        threshold: {
            "数量": data["count"],
            "占比": f"{data['count']/len(bonds)*100:.1f}%" if bonds else "0%",
            "示例": data["examples"]
        } for threshold, data in threshold_patterns.items()
    }
}

json_path = os.path.join(output_dir, DATA_CONFIG["output_json"])
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(detailed_results, f, ensure_ascii=False, indent=4)

print(f"详细分析结果已保存到: {json_path}")

In [ ]:
# 可视化结果
def create_summary_visualization(trigger_stats, mechanism_stats, feature_stats, output_dir):
    """
    创建分析结果的可视化图表
    """
    import matplotlib.pyplot as plt
    
    # 设置中文字体
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # 触发条件图表
    triggers = list(trigger_stats.keys())
    trigger_values = list(trigger_stats.values())
    axes[0].barh(triggers, trigger_values, color=VISUALIZATION_CONFIG["colors"][0])
    axes[0].set_title('加速到期触发条件分布', fontsize=VISUALIZATION_CONFIG["title_fontsize"])
    axes[0].set_xlabel('债券数量', fontsize=VISUALIZATION_CONFIG["label_fontsize"])
    
    # 保护机制图表
    mechanisms = list(mechanism_stats.keys())
    mechanism_values = list(mechanism_stats.values())
    axes[1].barh(mechanisms, mechanism_values, color=VISUALIZATION_CONFIG["colors"][1])
    axes[1].set_title('投资者保护机制分布', fontsize=VISUALIZATION_CONFIG["title_fontsize"])
    axes[1].set_xlabel('债券数量', fontsize=VISUALIZATION_CONFIG["label_fontsize"])
    
    # 其他特征图表
    features = list(feature_stats.keys())
    feature_values = list(feature_stats.values())
    axes[2].barh(features, feature_values, color=VISUALIZATION_CONFIG["colors"][2])
    axes[2].set_title('其他债券特征分布', fontsize=VISUALIZATION_CONFIG["title_fontsize"])
    axes[2].set_xlabel('债券数量', fontsize=VISUALIZATION_CONFIG["label_fontsize"])
    
    plt.tight_layout()
    
    # 保存图表
    chart_path = os.path.join(output_dir, "analysis_summary.png")
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    print(f"可视化图表已保存到: {chart_path}")
    
    plt.show()

# 执行可视化
try:
    create_summary_visualization(trigger_stats, mechanism_stats, feature_stats, output_dir)
except ImportError:
    print("matplotlib未安装，跳过可视化")
except Exception as e:
    print(f"可视化生成失败: {e}")

---

## 6. 工具函数

可复用的辅助函数集合。

In [ ]:
def match_patterns_in_text(text, patterns, context_length=50):
    """
    在文本中匹配多个模式，返回匹配结果和上下文
    
    参数:
        text: 待匹配的文本
        patterns: 正则表达式模式列表
        context_length: 上下文长度
    
    返回:
        匹配结果列表，每个元素包含(pattern, context)
    """
    matches = []
    for pattern in patterns:
        for match in re.finditer(pattern, text):
            start = max(0, match.start() - context_length)
            end = min(len(text), match.end() + context_length)
            context = text[start:end]
            matches.append({
                'pattern': pattern,
                'match': match.group(),
                'start': match.start(),
                'end': match.end(),
                'context': context
            })
    return matches

In [ ]:
def analyze_single_bond(clause, trigger_patterns=None, mechanism_patterns=None, feature_patterns=None):
    """
    分析单个债券的条款内容
    
    参数:
        clause: 条款文本
        trigger_patterns: 触发条件模式
        mechanism_patterns: 保护机制模式
        feature_patterns: 其他特征模式
    
    返回:
        分析结果字典
    """
    if trigger_patterns is None:
        trigger_patterns = TRIGGER_PATTERNS
    if mechanism_patterns is None:
        mechanism_patterns = PROTECTION_MECHANISM_PATTERNS
    if feature_patterns is None:
        feature_patterns = OTHER_FEATURE_PATTERNS
    
    result = {
        'triggers': [],
        'mechanisms': [],
        'features': []
    }
    
    # 分析触发条件
    for trigger, patterns in trigger_patterns.items():
        for pattern in patterns:
            if re.search(pattern, clause):
                result['triggers'].append(trigger)
                break
    
    # 分析保护机制
    for mechanism, patterns in mechanism_patterns.items():
        for pattern in patterns:
            if re.search(pattern, clause):
                result['mechanisms'].append(mechanism)
                break
    
    # 分析其他特征
    for feature, patterns in feature_patterns.items():
        for pattern in patterns:
            if re.search(pattern, clause):
                result['features'].append(feature)
                break
    
    return result

In [ ]:
def export_to_excel(bonds, trigger_stats, mechanism_stats, feature_stats, output_path):
    """
    导出分析结果到Excel文件
    
    参数:
        bonds: 债券信息列表
        trigger_stats: 触发条件统计
        mechanism_stats: 保护机制统计
        feature_stats: 其他特征统计
        output_path: 输出文件路径
    """
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        # 汇总统计
        summary_data = []
        for k, v in trigger_stats.items():
            summary_data.append({'类别': '触发条件', '特征': k, '数量': v})
        for k, v in mechanism_stats.items():
            summary_data.append({'类别': '保护机制', '特征': k, '数量': v})
        for k, v in feature_stats.items():
            summary_data.append({'类别': '其他特征', '特征': k, '数量': v})
        
        df_summary = pd.DataFrame(summary_data)
        df_summary.to_excel(writer, sheet_name='汇总统计', index=False)
        
        # 债券明细
        df_bonds = pd.DataFrame(bonds)
        df_bonds.to_excel(writer, sheet_name='债券明细', index=False)
    
    print(f"Excel报告已保存到: {output_path}")

In [ ]:
# 测试单个债券分析
if bonds:
    print("测试单个债券分析:")
    test_clause = bonds[0]['clause']
    test_result = analyze_single_bond(test_clause)
    print(f"债券代码: {bonds[0]['code']}")
    print(f"触发条件: {test_result['triggers']}")
    print(f"保护机制: {test_result['mechanisms']}")
    print(f"其他特征: {test_result['features']}")

In [ ]:
# 导出Excel报告
excel_path = os.path.join(output_dir, DATA_CONFIG["output_excel"])
export_to_excel(bonds, trigger_stats, mechanism_stats, feature_stats, excel_path)

---

## 7. 总结

本项目完成了信用债加速到期条款的全面分析，包括：

1. **触发条件识别**: 10种主要触发条件的识别和统计
2. **保护机制分析**: 5种投资者保护机制的分析
3. **其他特征统计**: 4种其他债券特征的统计
4. **分期还本模式**: 分期还本比例模式的提取
5. **交叉违约门槛**: 金额门槛的统计分析

### 输出文件

| 文件 | 说明 |
|------|------|
| 债券加速到期条款详细分析.json | 结构化分析结果 |
| 债券加速到期条款分析结果.csv | CSV格式分析结果 |
| 带加速到期条款存续个券_分析结果.xlsx | Excel格式完整报告 |
| analysis_summary.png | 可视化图表 |

### 注意事项

1. 条款解析依赖正则表达式，可能存在遗漏
2. 需要定期更新个券数据
3. 分析结果仅供参考，具体以募集说明书为准